In [2]:
from collections import defaultdict

def construir_arbol(aristas):
    arbol = defaultdict(list)
    for u, v in aristas:
        arbol[u].append(v)
        arbol[v].append(u)
    return arbol


def conjunto_independiente_maximo(arbol, raiz):
    padre = {raiz: None}  # Corrected to initialize as a dictionary
    orden = []
    pila = [raiz]
    visitado = {raiz}

    # Perform a DFS to build the parent dictionary and topological order
    while pila:
        nodo = pila.pop()
        orden.append(nodo)
        for vecino in arbol[nodo]:  # Iterate through neighbors
            if vecino not in visitado:
                visitado.add(vecino)
                padre[vecino] = nodo
                pila.append(vecino)

    dp0, dp1 = {}, {}

    # DP calculation from leaves up to the root
    for nodo in reversed(orden):
        suma_incluido = 1
        suma_no_incluido = 0
        for hijo in arbol[nodo]:
            if hijo != padre.get(nodo): # Use .get to handle root's parent
                # Ensure hijos are actual children in the DFS tree
                # rather than just neighbors in the general graph
                if padre.get(hijo) == nodo:
                    suma_incluido += dp0.get(hijo, 0) # Use .get with default 0 if not present yet
                    suma_no_incluido += max(dp0.get(hijo, 0), dp1.get(hijo, 0))

        dp1[nodo] = suma_incluido
        dp0[nodo] = suma_no_incluido

    tamaño_maximo = max(dp0[raiz], dp1[raiz])

    conjunto = set()
    decision = {raiz: dp1[raiz] >= dp0[raiz]}
    cola = [raiz]

    while cola:
        nodo = cola.pop()
        incluir = decision[nodo]
        if incluir:
            conjunto.add(nodo)
        for hijo in arbol[nodo]:
            if hijo != padre.get(nodo): # Use .get to handle root's parent
                if padre.get(hijo) == nodo: # Ensure hijos are actual children
                    if incluir:
                        decision[hijo] = False
                    else:
                        decision[hijo] = dp1[hijo] >= dp0[hijo]
                    cola.append(hijo)

    return tamaño_maximo, conjunto # Return statement moved inside the function



if __name__ == '__main__':

  aristas = [(1,2),(1,3),(2,4),(2,5)]
  arbol = construir_arbol(aristas)

  tamano, conjunto = conjunto_independiente_maximo(arbol, raiz=1) # Corrected typo in function call

  print(f"Tamaño del conjunto independiente maximo: {tamano}")
  print(f"Conjunto encontrado: {sorted(conjunto)}") # Corrected typo in print statement


Tamaño del conjunto independiente maximo: 3
Conjunto encontrado: [1, 4, 5]


In [4]:
from collections import defaultdict

def construir_grafo_bipartito(aristas):
    adyacencia = defaultdict(list)
    for u, v in aristas:
        adyacencia[u].append(v)
    return adyacencia

def intentar_emparejar(u, adyacencia, pareja_R, visitado):
    for v in adyacencia[u]:
        if v not in visitado:
            visitado.add(v)
            if v not in pareja_R or intentar_emparejar(pareja_R[v], adyacencia, pareja_R, visitado):
                pareja_R[v] = u
                return True
    return False


def matching_bipartito_maximo(adyacencia, vertices_L):
    pareja_R = {}
    tamano = 0

    for u in vertices_L:
        visitado = set()
        if intentar_emparejar(u, adyacencia, pareja_R, visitado):
            tamano += 1

    return tamano, pareja_R


if __name__ == '__main__':

  aristas = [
      ("a","x"),
      ("a","y"),
      ("a","z"),
      ("b","x"),
      ("b","y"),
      ("b","z"),
      ("c","x"),
      ("c","y"),
      ("c","z"),
  ]

  vertices_L = ["a","b","c"]

  adyacencia = construir_grafo_bipartito(aristas)
  tamano, pareja_R = matching_bipartito_maximo(adyacencia, vertices_L)

  print(f"Tamaño del matching maximo: {tamano}")
  print("emparejamiento")
  for v, u in sorted(pareja_R.items()): # Corrected to .items()
      print(f" {u} - {v}")


Tamaño del matching maximo: 3
emparejamiento
 c - x
 b - y
 a - z


In [6]:
def simplex(c,A,b):
    m = len(A)
    n = len(c)
    num_cols = n + m + 1

    tableau = []
    for i in range(m):
        fila = list(A[i]) + [1.0 if j == i else 0.0 for j in range(m)] + [float(b[i])]
        tableau.append(fila)

    fila_objetivo = [-float(coef) for coef in c] + [0.0] * m + [0.0]
    tableau.append(fila_objetivo)

    base = list(range(n, n+ m))

    while True:
        fila_obj = tableau[-1]
        col_entra = min(range(n + m), key = lambda j : fila_obj[j])
        if fila_obj[col_entra] >= -1e-9:
            break

        mejor_razon = None
        fila_sale = -1
        for i in range(m):
            coef = tableau[i][col_entra]
            if coef > 1e-9:
                razon = tableau[i][-1] / coef
                if mejor_razon is None or razon < mejor_razon - 1e-9:
                    mejor_razon = razon
                    fila_sale = i

        if fila_sale == -1:
            raise ValueError("el problema es no acotado")

        pivote = tableau[fila_sale][col_entra]
        tableau[fila_sale] = [v / pivote for v in tableau[fila_sale]]


        for i in range(len(tableau)):
            if i != fila_sale:
                factor = tableau[i][col_entra]
                if factor != 0:
                    tableau[i] = [
                        tableau[i][j] - factor * tableau[fila_sale][j]
                        for j in range(num_cols)
                    ]

        base[fila_sale] = col_entra


    solucion = [0.0] * n
    for i, var in enumerate(base):
        if var < n:
            solucion[var] = tableau[i][-1]

    valor_optimo = tableau[-1][-1]
    return valor_optimo, solucion


if __name__ == "__main__":


    c = [2,1]
    A = [
        [1,0],
        [0,1],
        [1,1],
    ]


    b = [4,3,5]


    valor_optimo, solucion = simplex(c,A,b)

    print(f"valor optimo {valor_optimo}") # Corrected variable name
    print(f"x = {solucion[0]}, y = {solucion[1]}") # Corrected f-string syntax


valor optimo 9.0
x = 4.0, y = 1.0
